# Kazan housing market: data preparation

This notebook extracts residential-premises records for Kazan from four nationwide quarterly Rosreestr files and creates the local dataset used in subsequent analysis. Source files are processed incrementally to keep memory usage bounded.

In [3]:
from pathlib import Path

import pandas as pd


# This works whether Jupyter starts in the repository root or in notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_FILE = DATA_DIR / "kazan_residential_transactions_2025_raw.csv"
KAZAN_OKATO_PREFIX = "92401"
PREMISES_CODE = "002001003000"
RESIDENTIAL_PURPOSE_CODE = "206002000000"
CHUNK_SIZE = 100_000

## Source files

The analysis expects one nationwide source file for each quarter of 2025.

In [4]:
csv_files = sorted(DATA_DIR.glob("dataset_СДЕЛКИ_*.csv"))

if len(csv_files) != 4:
    raise ValueError(
        f"Expected 4 quarterly files, but found {len(csv_files)} in {DATA_DIR}"
    )

for file_path in csv_files:
    print(file_path.name)

dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_2.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_3.csv
dataset_СДЕЛКИ_r-r_01-92_y_2025_q_4.csv


## Extract Kazan residential-premises records

Kazan is selected using the `92401` OKATO prefix. The scope is then restricted to objects classified as premises (`002001003000`) with residential purpose (`206002000000`). This is the closest available proxy for apartments; the source does not provide a separate field that reliably distinguishes apartments from rooms. The generated file is replaced on each complete notebook run.

In [ ]:
summary_rows = []
write_header = True

for file_path in csv_files:
    raw_row_count = 0
    residential_row_count = 0
    quarter = f"Q{file_path.stem[-1]}"

    for chunk in pd.read_csv(
        file_path,
        sep="~",
        encoding="utf-8",
        dtype="string",
        chunksize=CHUNK_SIZE,
    ):
        raw_row_count += len(chunk)

        is_kazan = chunk["okato"].str.startswith(
            KAZAN_OKATO_PREFIX,
            na=False,
        )
        is_residential_premises = (
            (chunk["realestate_type_code"] == PREMISES_CODE)
            & (chunk["purpose_code"] == RESIDENTIAL_PURPOSE_CODE)
        )
        target_rows = is_kazan & is_residential_premises
        residential_chunk = chunk.loc[target_rows].copy()
        residential_row_count += len(residential_chunk)

        if len(residential_chunk) > 0:
            residential_chunk["quarter"] = quarter

            residential_chunk.to_csv(
                OUTPUT_FILE,
                mode="w" if write_header else "a",
                header=write_header,
                index=False,
                encoding="utf-8",
            )
            write_header = False

    summary_rows.append(
        {
            "quarter": quarter,
            "raw_rows": raw_row_count,
            "residential_rows": residential_row_count,
        }
    )

load_summary = pd.DataFrame(summary_rows)
load_summary

,quarter,raw_rows,kazan_rows
0,Q1,542271,2662
1,Q2,638474,2451
2,Q3,713281,2853
3,Q4,777632,3329


## Validate the saved dataset

The saved dataset is checked for its expected dimensions, quarterly coverage, geographic scope, object type, and residential purpose.

In [ ]:
residential_raw = pd.read_csv(
    OUTPUT_FILE,
    dtype="string",
    encoding="utf-8",
)

non_kazan_rows = (
    ~residential_raw["okato"].str.startswith(KAZAN_OKATO_PREFIX, na=False)
).sum()
non_premises_rows = (
    residential_raw["realestate_type_code"] != PREMISES_CODE
).sum()
non_residential_rows = (
    residential_raw["purpose_code"] != RESIDENTIAL_PURPOSE_CODE
).sum()

print(f"Saved file: {OUTPUT_FILE}")
print(f"Dataset shape: {residential_raw.shape}")
print(f"Non-Kazan rows: {non_kazan_rows}")
print(f"Non-premises rows: {non_premises_rows}")
print(f"Non-residential rows: {non_residential_rows}")
print()
print("Rows by quarter:")
print(residential_raw["quarter"].value_counts().sort_index())

residential_raw.head()

Saved file: c:\Users\Artem\Documents\Kazan_Housing\Kazan-housing\data\kazan_transactions_2025_raw.csv
Dataset shape: (11295, 19)
Non-Kazan rows: 0

Rows by quarter:
quarter
Q4    3329
Q3    2853
Q1    2662
Q2    2451
Name: count, dtype: Int64


,number,okato,region_code,district,city,quarter_cad_number,street,realestate_type_code,wall_material_code,year_build,floor,purpose_code,area,period_start_date,deal_price,currency,doc_type,quarter,source_file
0,1,92401385000,16,<NA>,Казань,16:50:250101,Дорожная (Киндери),002001002000,<NA>,<NA>,1,204006000000,24.4,2025-01-01,67000.0,рубль,ДКП,Q1,dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
1,1,92401380000,16,<NA>,Казань,16:50:160205,Хусаина Мавлютова,002001003000,061001001001,1986,9,206002000000,56.9,2025-01-01,8900000.0,рубль,ДКП,Q1,dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
2,3,92401367000,16,<NA>,Казань,16:50:011102,Старая,002001003000,<NA>,<NA>,<NA>,206002000000,7991.3099999999995,2025-01-01,54923916,рубль,ДДУ,Q1,dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
3,1,92401367000,16,<NA>,Казань,16:50:011705,Николая Столбова,002001003000,061001003000,2020,2,206002000000,62.1,2025-01-01,26400000,рубль,ДКП,Q1,dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
4,1,92401385000,16,<NA>,Казань,16:50:060510,Комарова,002001003000,061001007001,1969,5,206002000000,59.4,2025-01-01,9000000.0,рубль,ДКП,Q1,dataset_СДЕЛКИ_r-r_01-92_y_2025_q_1.csv
